# AutoSQUID — measurement cycle (example)

In [ ]:
import AutoSQUID as sq

# For complete config list and descriptions, see AutoSQUID/config.py.
cfg = sq.Config(
    scan_interval_s=[100e-6],
    temp_label="auto",         # "auto" reads+rounds the MXC temp; or hard-set e.g. "14mK" if temp is unstable.
    n_trials=2,
    port="COM3",
    temp_logger=True,          # set False to skip the background MXC temperature thread (+ per-trace TEMP csv)
    live_jump_check=True,      # set False to drain the run without the mid-run jump/rail abort (post-hoc gate still runs)
    data_root="",          # set
    user="",               # set
    date=""                # set
)
cfg.temp_reader = None     # set: fn(channel) -> T in K

In [ ]:
dev, cfg.daq_ai = sq.detect_ai_channel(cfg)
print(f"Using DAQ AI channel {cfg.daq_ai} for SQUID readout (device {dev})")

In [ ]:
sq.s_lock(cfg)                                   # lock the input SQUID (or do it in PCS102DA software)
res = sq.auto_s_tune(cfg, start_sflux=50.0, target_V=0.0, tol_V=0.020)
print(res)
if res["status"] != "converged":
    print("NOT centered — nudge S-flux manually closer / check lock, then re-run.")

In [ ]:
import datetime
cfg.outdir.mkdir(parents=True, exist_ok=True)
print(f"temperature now: {sq.read_temp(cfg):.4f} K")
print(f"temperature label for this run: {sq.resolve_temp_label(cfg)}")
print(f"measurement START : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
for tau in cfg.scan_intervals:
    if sq.run_cycle(cfg, tau) == "reset_fail":
        break